# ⚙️ Actividad 09: Pipeline ETL Completo
---
**Entrada:** `data/02_interim/dataset_integrado.csv`  
**Salida:** `data/03_processed/master_dataset_fase1_v2.csv` + carga en PostgreSQL

Pasos:
1. Feature Engineering (codificación cíclica)
2. Escalamiento con StandardScaler
3. Carga en PostgreSQL (dim_tiempo → dim_ubicacion → fact_produccion_limon)
4. Exportación CSV final


In [ ]:

import os, sys, json, warnings
import numpy as np, pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine, text
warnings.filterwarnings('ignore')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))
with open('data/02_interim/pipeline_config.json','r',encoding='utf-8') as f:
    CFG = json.load(f)
DIRS=CFG['DIRS']; INTERIM=DIRS['interim']; PROCESSED=DIRS['processed']
SCALERS=DIRS['scalers']; PG_URI=CFG['PG_URI']
print(f"CWD: {os.getcwd()} | Config OK")


## 9.1 Extraer Dataset Integrado

In [ ]:

df = pd.read_csv(f"{INTERIM}/dataset_integrado.csv")
print(f"Shape: {df.shape}")
print(f"Rango: {df['fecha_evento'].min()} → {df['fecha_evento'].max()}")
print(f"Columnas: {df.columns.tolist()}")
df.head(3)


## 9.2 Feature Engineering — Codificación Cíclica

In [ ]:

df['fecha_dt'] = pd.to_datetime(df['fecha_evento'])
df['anho']      = df['fecha_dt'].dt.year
df['mes']       = df['fecha_dt'].dt.month
df['trimestre'] = df['fecha_dt'].dt.quarter
df['month_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
df = df.drop(columns=['fecha_dt'])
print("Codificación cíclica aplicada: month_sin, month_cos")
print(f"Trimestres: {sorted(df['trimestre'].unique())}")


## 9.3 Escalamiento con StandardScaler

In [ ]:

FEATS = ['produccion_t','cosecha_ha','precio_chacra_kg',
         'num_emergencias','total_afectados','has_cultivo_perdidas','n_noticias',
         'T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'ALLSKY_SFC_SW_DWN']
cols = [c for c in FEATS if c in df.columns]

scaler = StandardScaler()
df[cols] = scaler.fit_transform(df[cols].fillna(0))

scaler_path = f"{SCALERS}/scaler_fase1_v2.pkl"
joblib.dump(scaler, scaler_path)
print(f"Variables escaladas: {cols}")
print(f"Scaler guardado: {scaler_path}")


## 9.4 Carga en PostgreSQL

In [ ]:

COORDS = {
    'AMAZONAS':(-6.23,-77.87),'ANCASH':(-9.53,-77.53),'APURIMAC':(-13.64,-72.88),
    'AREQUIPA':(-16.41,-71.54),'AYACUCHO':(-13.16,-74.22),'CAJAMARCA':(-7.16,-78.50),
    'CALLAO':(-12.06,-77.15),'CUSCO':(-13.53,-71.97),'HUANCAVELICA':(-12.78,-74.97),
    'HUANUCO':(-9.93,-76.24),'ICA':(-14.07,-75.73),'JUNIN':(-11.16,-75.00),
    'LA LIBERTAD':(-8.11,-79.03),'LAMBAYEQUE':(-6.77,-79.84),'LIMA':(-12.05,-77.04),
    'LORETO':(-3.75,-73.25),'MADRE DE DIOS':(-12.59,-69.19),'MOQUEGUA':(-17.19,-70.93),
    'PASCO':(-10.69,-76.26),'PIURA':(-5.19,-80.63),'PUNO':(-15.84,-70.02),
    'SAN MARTIN':(-6.52,-76.36),'TACNA':(-18.01,-70.25),'TUMBES':(-3.57,-80.45),
    'UCAYALI':(-8.38,-74.54),
}

try:
    engine = create_engine(PG_URI)
    with engine.connect() as conn:
        conn.execute(text("TRUNCATE TABLE fact_produccion_limon RESTART IDENTITY CASCADE"))
        conn.execute(text("TRUNCATE TABLE dim_tiempo RESTART IDENTITY CASCADE"))
        conn.execute(text("TRUNCATE TABLE dim_ubicacion RESTART IDENTITY CASCADE"))
        conn.execute(text("TRUNCATE TABLE dim_clima RESTART IDENTITY CASCADE"))
        conn.execute(text("TRUNCATE TABLE dim_multimodal RESTART IDENTITY CASCADE"))
        conn.commit()

    # 1. dim_tiempo
    dim_t = df[['fecha_evento','anho','mes','trimestre','month_sin','month_cos']].drop_duplicates('fecha_evento')
    dim_t.to_sql('dim_tiempo', engine, if_exists='append', index=False, method='multi', chunksize=500)
    print(f"  [OK] dim_tiempo: {len(dim_t)} registros")

    # 2. dim_ubicacion
    dim_u = df[['departamento','provincia']].drop_duplicates().reset_index(drop=True)
    dim_u['lat'] = dim_u['departamento'].map(lambda d: COORDS.get(d,(None,None))[0])
    dim_u['lon'] = dim_u['departamento'].map(lambda d: COORDS.get(d,(None,None))[1])
    dim_u.to_sql('dim_ubicacion', engine, if_exists='append', index=False, method='multi', chunksize=500)
    print(f"  [OK] dim_ubicacion: {len(dim_u)} registros")

    # 3. dim_clima
    clima_cols = {
        'T2M_MAX': 'temp_max_c', 'T2M_MIN': 'temp_min_c', 
        'PRECTOTCORR': 'precipitacion_mm', 'RH2M': 'humedad_rel_pct',
        'ALLSKY_SFC_SW_DWN': 'radiacion_solar'
    }
    dim_c = df[list(clima_cols.keys())].rename(columns=clima_cols)
    dim_c.to_sql('dim_clima', engine, if_exists='append', index=False, method='multi', chunksize=500)
    print(f"  [OK] dim_clima: {len(dim_c)} registros")

    # 4. dim_multimodal
    multi_cols = {
        'n_noticias': 'n_noticias', 'num_emergencias': 'num_emergencias',
        'total_afectados': 'total_afectados'
    }
    dim_m = df[list(multi_cols.keys())].rename(columns=multi_cols)
    dim_m['avg_sentimiento'] = 0.0 # Placeholder Fase 2
    dim_m.to_sql('dim_multimodal', engine, if_exists='append', index=False, method='multi', chunksize=500)
    print(f"  [OK] dim_multimodal: {len(dim_m)} registros")

    # 5. fact_produccion_limon
    with engine.connect() as conn:
        dt_map = pd.read_sql('SELECT id_tiempo, fecha_evento FROM dim_tiempo', conn)
        du_map = pd.read_sql('SELECT id_ubicacion, departamento, provincia FROM dim_ubicacion', conn)
        # Como dim_clima y dim_multimodal son 1:1 con las filas del dataframe original (por ahora), 
        # podemos usar los IDs generados secuencialmente.
        dc_ids = pd.read_sql('SELECT id_clima FROM dim_clima ORDER BY id_clima', conn)
        dm_ids = pd.read_sql('SELECT id_multimodal FROM dim_multimodal ORDER BY id_multimodal', conn)

    df_f = df.merge(dt_map, on='fecha_evento').merge(du_map, on=['departamento','provincia'])
    df_f['id_clima'] = dc_ids['id_clima']
    df_f['id_multimodal'] = dm_ids['id_multimodal']
    
    fact_cols = ['id_tiempo','id_ubicacion','id_clima','id_multimodal','produccion_t','cosecha_ha','precio_chacra_kg']
    df_load = df_f[fact_cols].dropna(subset=['id_tiempo','id_ubicacion'])
    
    df_load.to_sql('fact_produccion_limon', engine, if_exists='append', index=False, method='multi', chunksize=500)
    print(f"  [OK] fact_produccion_limon: {len(df_load):,} registros")
    engine.dispose()
except Exception as e:
    print(f"  [ERROR PostgreSQL] {e}")
    print("  Continuando sin carga en BD...")


## 9.5 Exportar CSV Final

In [ ]:

out = f"{PROCESSED}/master_dataset_fase1_v2.csv"
df.to_csv(out, index=False, encoding='utf-8-sig')
print(f"Shape final: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
print(f"[OK] {out}")
print("\n[ACTIVIDAD 09] COMPLETADA.")


# Actividad 09 Finalizada OK
